# LEI Cluster Analysis (AWS Embeddings)

This notebook runs the same AWS embedding + PCA + KMeans workflow, focused on the LEI subset table (`PRODUCTS_LEI`).

It performs one embedding pass, then evaluates K candidates:
- `k in [3, 5, 10, 15, 20, 30, 40]`

After choosing the final L5 `k`, it can run LLM-assisted labeling and publish labels to `LEI_PROPOSED_L5`.

In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import silhouette_score

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    raise FileNotFoundError("Could not locate project root containing product_classifier_utils.py")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import (  # noqa: E402
    build_product_text,
    embed_texts_from_cache,
    get_bedrock_client,
    get_snowflake_session,
    load_product_data,
    stable_text_hash,
)

In [24]:
# Core run configuration
TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LEI"
DATASET_SLUG = TABLE.split(".")[-1].lower()
LABEL_COLUMN = "PARENT_3_CATEGORY"
MIN_CATEGORY_COUNT = 1
ONLY_PRICED = False
EXCLUDE_INSERT_PRODUCTS = True
ROW_LIMIT = None

# Embedding + cache settings
AWS_PROFILE = "staging.admin"
AWS_REGION = "us-east-1"
MODEL_ID = "amazon.titan-embed-text-v1"
MAX_WORKERS = 16
CHECKPOINT_EVERY = 2500
MAX_RETRIES = 5
CACHE_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

# PCA + K exploration settings
PCA_COMPONENTS_TO_TRY = [256]
K_VALUES_L4 = [3, 4,5,6,7,8,9]
K_VALUES_L5 = [10,11,12,13]
K_VALUES_ALL = sorted(set(K_VALUES_L4 + K_VALUES_L5))
SAMPLE_SIZE_FOR_SILHOUETTE = 30000
RANDOM_STATE = 42

# Optional override controls for business-selected configs.
# Use None to keep score-based auto-selection.
OVERRIDE_L4_K = None       # e.g., 6
OVERRIDE_L5_K = None       # e.g., 48
OVERRIDE_L4_PCA = 256      # fixed for consistency with LCG final run
OVERRIDE_L5_PCA = 256      # fixed for consistency with LCG final run
FORCE_SHARED_PCA = 256     # enforce one PCA across L4 and L5

# Output settings
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "analysis"
RUN_TAG = "lei_aws_cluster_experiment"
SAVE_TO_SNOWFLAKE_TABLE = False
OUTPUT_TABLE_NAME = "PROPOSED_LEI_AWS"

# Optional LLM labeling
RUN_LLM_LABELING = True
LLM_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
LLM_MAX_CATEGORIES = 12
LLM_TEMPERATURE = 0.2

In [5]:
sf_session = get_snowflake_session()
df = load_product_data(
    session=sf_session,
    table=TABLE,
    label_column=LABEL_COLUMN,
    min_category_count=MIN_CATEGORY_COUNT,
    row_limit=ROW_LIMIT,
)

if ONLY_PRICED:
    df = df[df["PRICING_STATUS_C"].astype(str).str.upper() == "PRICED"].reset_index(drop=True)
if EXCLUDE_INSERT_PRODUCTS:
    before_rows = len(df)
    df = df[~df["DESCRIPTION"].fillna("").astype(str).str.upper().str.contains("INSERT", regex=False)].reset_index(drop=True)
    print(f"Excluded INSERT-description rows: {before_rows - len(df):,}")

required = {"PRODUCT_ID", "PRODUCT_NAME", "DESCRIPTION", "PRICING_STATUS_C", "LIST_PRICE_C"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns for text embedding: {missing}")

print(f"Loaded rows: {len(df):,}")
print(df[["PRODUCT_ID", "PRICING_STATUS_C", LABEL_COLUMN]].head(3))

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZLdb9owFMX%2Flch7TuzwjQVUtCwaVWkZ0G3am5tcqJXETn0dEvjr5%2FAxdQ%2BttLfIOce%2F43vu6KbOM28PBqVWYxIGjHigYp1ItRuT503kD4iHVqhEZFrBmBwAyc1khCLPCj4t7atawVsJaD13kULe%2FBiT0iiuBUrkSuSA3MZ8PV088FbAuEAEYx2OXCwJSsd6tbbglFZVFVTtQJsdbTHGKBtSp2okX8g7RPE5ozDa6lhnV0vt3vQBIqSs0yCcwhGWF%2BOtVOcRfEZ5OYuQf9tslv7yab0h3vT6ujutsMzBrMHsZQzPq4dzAHQJijTu9jvdQVCiDwKtHwaodLXNRAqxzovSumsD90W3kNBM76Qb1nw2JkUqk2Oip%2B09W0X518PPamCqQ8kG0XD1Eq%2Fvo%2B8ir%2FKFiH71Ho%2F3nZh4P67Vtppq54glzFVTqHVHrNXzWdcP2SYc8u6At3pBv939TbyZK1QqYU%2FOa2qMpRsSQB2%2FCrWDQKdWnEKKoqB%2F81OoU3uU9VteL%2FrQT5Pb3rBPETVteiPn1eGnIGby3wMZ0ff2yxo%2Bumbms6XOZHzwIm1yYT8uLgzC04lM%2FO1JyiEXMpsmiQFEV2CW6erOgLBu260pgdDJmfrvvk%2F%2BAA%3D%3D&RelayState=ver%3A3-hint%3A9376132679206278-ETMsDgAAAZ4TeDmvABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEGHHOTVTU7ZVl%2F2DGLLYdRsAAACg4cy

In [10]:
def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, dict):
        raise ValueError(f"Cache file {path} is not a dict")
    return obj


def save_cache(path: Path, cache: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(cache, f)


text_series = build_product_text(df)
texts = text_series.tolist()
text_hashes = [stable_text_hash(t) for t in texts]

cache = load_cache(CACHE_PATH)
cache_before = len(cache)
client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)


def checkpoint_cb(cache_obj: dict, processed_count: int) -> None:
    save_cache(CACHE_PATH, cache_obj)
    print(f"Checkpointed {processed_count:,} new embeddings. Cache size={len(cache_obj):,}")


X = embed_texts_from_cache(
    texts=texts,
    text_hashes=text_hashes,
    cache=cache,
    client=client,
    model_id=MODEL_ID,
    show_progress=True,
    max_workers=MAX_WORKERS,
    checkpoint_every=CHECKPOINT_EVERY if CHECKPOINT_EVERY > 0 else None,
    on_checkpoint=checkpoint_cb if CHECKPOINT_EVERY > 0 else None,
    max_retries=MAX_RETRIES,
)

save_cache(CACHE_PATH, cache)
print(f"Embedding matrix shape: {X.shape}")
print(f"Cache size: {cache_before:,} -> {len(cache):,}")

df["TEXT_TO_EMBED"] = text_series

Embedding matrix shape: (527186, 1536)
Cache size: 1,805,266 -> 1,805,266


In [11]:
def find_local_maxima(score_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pca_label, grp in score_df.groupby("pca"):
        g = grp.sort_values("k").reset_index(drop=True)
        for i in range(len(g)):
            left = g.loc[i - 1, "silhouette"] if i > 0 else -np.inf
            cur = g.loc[i, "silhouette"]
            right = g.loc[i + 1, "silhouette"] if i < len(g) - 1 else -np.inf
            if cur >= left and cur >= right:
                rows.append(g.loc[i].to_dict())
    if not rows:
        return pd.DataFrame(columns=score_df.columns)
    return pd.DataFrame(rows).sort_values("silhouette", ascending=False).reset_index(drop=True)


rng = np.random.default_rng(RANDOM_STATE)
sample_n = min(SAMPLE_SIZE_FOR_SILHOUETTE, len(X))
sample_idx = rng.choice(len(X), size=sample_n, replace=False)

rows = []
transformed_by_pca = {}
pca_model_by_label = {}

for pca_components in PCA_COMPONENTS_TO_TRY:
    if pca_components is None:
        X_eval = X
        pca_label = "none"
        explained = np.nan
        pca_model = None
    else:
        pca_model = PCA(n_components=pca_components, random_state=RANDOM_STATE)
        X_eval = pca_model.fit_transform(X).astype(np.float32)
        pca_label = str(pca_components)
        explained = float(np.sum(pca_model.explained_variance_ratio_))

    transformed_by_pca[pca_label] = X_eval
    pca_model_by_label[pca_label] = pca_model

    X_sample = X_eval[sample_idx]
    for k in K_VALUES_ALL:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto", max_iter=300)
        labels = km.fit_predict(X_sample)
        sil = silhouette_score(X_sample, labels)
        rows.append(
            {
                "pca": pca_label,
                "pca_components": pca_components,
                "k": int(k),
                "silhouette": float(sil),
                "explained_variance_sum": explained,
                "range": "l4" if k in K_VALUES_L4 else "l5",
            }
        )

results_df = pd.DataFrame(rows).sort_values("silhouette", ascending=False).reset_index(drop=True)
results_df.head(20)

/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/e

,pca,pca_components,k,silhouette,explained_variance_sum,range
0,256,256,3,0.065904,0.966485,l4
1,256,256,5,0.055318,0.966485,l4
2,256,256,13,0.047549,0.966485,l5
3,256,256,12,0.047412,0.966485,l5
4,256,256,11,0.045684,0.966485,l5
5,256,256,7,0.045237,0.966485,l4
6,256,256,9,0.043221,0.966485,l4
7,256,256,10,0.043093,0.966485,l5
8,256,256,4,0.042022,0.966485,l4
9,256,256,8,0.041373,0.966485,l4


In [12]:
results_l4 = results_df[results_df["range"] == "l4"].copy()
results_l5 = results_df[results_df["range"] == "l5"].copy()

local_max_l4 = find_local_maxima(results_l4)
local_max_l5 = find_local_maxima(results_l5)

print("Top L4-range results:")
display(results_l4.head(12))

print("\nLocal maxima in L4-range:")
display(local_max_l4.head(12))

print("\nTop L5-range results:")
display(results_l5.head(12))

print("\nLocal maxima in L5-range:")
display(local_max_l5.head(12))

Top L4-range results:


,pca,pca_components,k,silhouette,explained_variance_sum,range
0,256,256,3,0.065904,0.966485,l4
1,256,256,5,0.055318,0.966485,l4
5,256,256,7,0.045237,0.966485,l4
6,256,256,9,0.043221,0.966485,l4
8,256,256,4,0.042022,0.966485,l4
9,256,256,8,0.041373,0.966485,l4
10,256,256,6,0.039855,0.966485,l4



Local maxima in L4-range:


,pca,pca_components,k,silhouette,explained_variance_sum,range
0,256,256,3,0.065904,0.966485,l4
1,256,256,5,0.055318,0.966485,l4
2,256,256,7,0.045237,0.966485,l4
3,256,256,9,0.043221,0.966485,l4



Top L5-range results:


,pca,pca_components,k,silhouette,explained_variance_sum,range
2,256,256,13,0.047549,0.966485,l5
3,256,256,12,0.047412,0.966485,l5
4,256,256,11,0.045684,0.966485,l5
7,256,256,10,0.043093,0.966485,l5



Local maxima in L5-range:


,pca,pca_components,k,silhouette,explained_variance_sum,range
0,256,256,13,0.047549,0.966485,l5


In [13]:
# Choose final configs (defaults to best silhouette in each range),
# with optional explicit overrides for business-selected values.
BEST_L4 = local_max_l4.iloc[0] if len(local_max_l4) else results_l4.iloc[0]
BEST_L5 = local_max_l5.iloc[0] if len(local_max_l5) else results_l5.iloc[0]


def normalize_pca_value(v):
    if v is None:
        return "none"
    s = str(v).strip().lower()
    if s in {"none", "null", ""}:
        return "none"
    return str(int(float(s)))


FINAL_PCA_L4 = str(BEST_L4["pca"])
FINAL_K_L4 = int(BEST_L4["k"])
FINAL_PCA_L5 = str(BEST_L5["pca"])
FINAL_K_L5 = int(BEST_L5["k"])

if OVERRIDE_L4_PCA is not None:
    FINAL_PCA_L4 = normalize_pca_value(OVERRIDE_L4_PCA)
if OVERRIDE_L5_PCA is not None:
    FINAL_PCA_L5 = normalize_pca_value(OVERRIDE_L5_PCA)
if FORCE_SHARED_PCA is not None:
    shared = normalize_pca_value(FORCE_SHARED_PCA)
    FINAL_PCA_L4 = shared
    FINAL_PCA_L5 = shared

if OVERRIDE_L4_K is not None:
    FINAL_K_L4 = int(OVERRIDE_L4_K)
if OVERRIDE_L5_K is not None:
    FINAL_K_L5 = int(OVERRIDE_L5_K)

if FINAL_PCA_L4 not in transformed_by_pca:
    raise ValueError(f"FINAL_PCA_L4={FINAL_PCA_L4} not found. Add it to PCA_COMPONENTS_TO_TRY and rerun PCA sweep.")
if FINAL_PCA_L5 not in transformed_by_pca:
    raise ValueError(f"FINAL_PCA_L5={FINAL_PCA_L5} not found. Add it to PCA_COMPONENTS_TO_TRY and rerun PCA sweep.")

print(
    "Chosen L4 config:",
    {
        "pca": FINAL_PCA_L4,
        "k": FINAL_K_L4,
        "silhouette_auto_candidate": float(BEST_L4["silhouette"]),
        "manual_override_applied": bool(OVERRIDE_L4_K is not None or OVERRIDE_L4_PCA is not None or FORCE_SHARED_PCA is not None),
    },
)
print(
    "Chosen L5 config:",
    {
        "pca": FINAL_PCA_L5,
        "k": FINAL_K_L5,
        "silhouette_auto_candidate": float(BEST_L5["silhouette"]),
        "manual_override_applied": bool(OVERRIDE_L5_K is not None or OVERRIDE_L5_PCA is not None or FORCE_SHARED_PCA is not None),
    },
)

# Fit final models on full set
X_l4 = transformed_by_pca[FINAL_PCA_L4]
X_l5 = transformed_by_pca[FINAL_PCA_L5]

km_l4 = KMeans(n_clusters=FINAL_K_L4, random_state=RANDOM_STATE, n_init="auto", max_iter=300)
km_l5 = KMeans(n_clusters=FINAL_K_L5, random_state=RANDOM_STATE, n_init="auto", max_iter=300)

df["CLUSTER_L4"] = km_l4.fit_predict(X_l4).astype("int32")
df["CLUSTER_L5_EXPLORATORY"] = km_l5.fit_predict(X_l5).astype("int32")

display(df[["CLUSTER_L4", "CLUSTER_L5_EXPLORATORY"]].describe())

Chosen L4 config: {'pca': '256', 'k': 3, 'silhouette_auto_candidate': 0.06590379029512405, 'manual_override_applied': True}
Chosen L5 config: {'pca': '256', 'k': 13, 'silhouette_auto_candidate': 0.047549307346343994, 'manual_override_applied': True}


/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/clus

,CLUSTER_L4,CLUSTER_L5_EXPLORATORY
count,527186.000000,527186.000000
mean,0.722468,5.790214
std,0.740608,3.763302
min,0.000000,0.000000
25%,0.000000,2.000000
50%,1.000000,6.000000
75%,1.000000,9.000000
max,2.000000,12.000000


In [14]:
# Cluster interpretation terms for both L4 and L5 cluster outputs.
DOMAIN_STOPWORDS = {
    "price", "pricing", "status", "quote", "quoted", "description", "insert",
    "item", "items", "product", "products", "unit", "units", "DWK", "sizing", 
    "packaging", "quantity", "quantities", "size", "sizes", "available", "availability", 
    "capacity", "capacities", "volume", "volumes", "length", "lengths", "width", "widths", 
    "height", "heights", "depth", "depths", "diameter", "diameters", "radius", "radii", 
    "weight", "weights"
}
STOPWORDS = list(ENGLISH_STOP_WORDS.union(DOMAIN_STOPWORDS))

vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words=STOPWORDS,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
)

tfidf_matrix = vectorizer.fit_transform(df["TEXT_TO_EMBED"])
feature_names = np.array(vectorizer.get_feature_names_out())


def top_terms_per_cluster(tfidf, dataf, cluster_col, features, top_n=15):
    out = {}
    for c in sorted(dataf[cluster_col].unique()):
        mask = (dataf[cluster_col] == c).values
        centroid = np.asarray(tfidf[mask].mean(axis=0)).ravel()
        top_idx = centroid.argsort()[::-1][:top_n]
        out[int(c)] = features[top_idx].tolist()
    return out


cluster_keywords_l4 = top_terms_per_cluster(tfidf_matrix, df, "CLUSTER_L4", feature_names, top_n=15)
cluster_keywords_l5 = top_terms_per_cluster(tfidf_matrix, df, "CLUSTER_L5_EXPLORATORY", feature_names, top_n=15)

print("Sample L4 cluster terms:")
display(dict(list(cluster_keywords_l4.items())[:3]))

print("Sample L5 cluster terms:")
display(dict(list(cluster_keywords_l5.items())[:3]))

/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['dwk'] not in stop_words.
  warnings.warn(


Sample L4 cluster terms:


{0: ['c09',
  '1g',
  '1g priced',
  '5g',
  '5g priced',
  '25g',
  '25g priced',
  'acid',
  'acid c09',
  'methyl',
  '100g',
  '100g priced',
  'bromo',
  '63',
  '10g'],
 1: ['c09',
  '250mg',
  '250mg priced',
  '100mg',
  '100mg priced',
  '5mg',
  '5mg priced',
  '25mg',
  '25mg priced',
  '50mg',
  '50mg priced',
  '10mg',
  '10mg priced',
  'hydrochloride',
  '500mg'],
 2: ['number',
  'manufacturer',
  'manufacturer number',
  'pk',
  'pk manufacturer',
  'pc',
  'url',
  'com',
  'https',
  'url https',
  'hazmat',
  'shipping',
  'requires',
  'requires list',
  'cenmed']}

Sample L5 cluster terms:


{0: ['c09',
  '1ml priced',
  '1ml',
  '25g',
  '25g priced',
  '1g',
  '1g priced',
  '5g',
  '5g priced',
  'acid',
  '100g',
  '100g priced',
  '63',
  'acid c09',
  '38'],
 1: ['1g',
  '1g priced',
  'c09',
  '5g',
  '5g priced',
  'acid',
  'methyl',
  'acid c09',
  '25g',
  '25g priced',
  'amino',
  'tert',
  'yl',
  'carboxylate',
  'butyl'],
 2: ['gloves',
  'pk',
  'pc',
  'pk manufacturer',
  'manufacturer number',
  'pc pk',
  'cut',
  'number',
  'manufacturer',
  'resistant',
  'ansi',
  'resistant gloves',
  'puncture',
  'moq',
  'pkg']}

In [15]:
out_dir = LOCAL_OUTPUT_DIR / RUN_TAG
out_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(out_dir / f"{DATASET_SLUG}_pca_k_silhouette_all.csv", index=False)
results_l4.to_csv(out_dir / f"{DATASET_SLUG}_pca_k_silhouette_l4_range.csv", index=False)
results_l5.to_csv(out_dir / f"{DATASET_SLUG}_pca_k_silhouette_l5_range.csv", index=False)
local_max_l4.to_csv(out_dir / f"{DATASET_SLUG}_local_maxima_l4_range.csv", index=False)
local_max_l5.to_csv(out_dir / f"{DATASET_SLUG}_local_maxima_l5_range.csv", index=False)

(df[["PRODUCT_ID", LABEL_COLUMN, "PRICING_STATUS_C", "CLUSTER_L4", "CLUSTER_L5_EXPLORATORY", "TEXT_TO_EMBED"]]
   .to_csv(out_dir / f"{DATASET_SLUG}_cluster_assignments.csv", index=False))

pd.DataFrame(
    [{"cluster_l4": c, "top_terms": ", ".join(terms)} for c, terms in cluster_keywords_l4.items()]
).to_csv(out_dir / f"{DATASET_SLUG}_cluster_l4_top_terms.csv", index=False)

pd.DataFrame(
    [{"cluster_l5": c, "top_terms": ", ".join(terms)} for c, terms in cluster_keywords_l5.items()]
).to_csv(out_dir / f"{DATASET_SLUG}_cluster_l5_top_terms.csv", index=False)

run_config = {
    "table": TABLE,
    "label_column": LABEL_COLUMN,
    "min_category_count": MIN_CATEGORY_COUNT,
    "only_priced": ONLY_PRICED,
    "row_limit": ROW_LIMIT,
    "embedding_model_id": MODEL_ID,
    "final_l4": {"pca": FINAL_PCA_L4, "k": FINAL_K_L4},
    "final_l5": {"pca": FINAL_PCA_L5, "k": FINAL_K_L5},
    "override_inputs": {
        "override_l4_k": OVERRIDE_L4_K,
        "override_l5_k": OVERRIDE_L5_K,
        "override_l4_pca": OVERRIDE_L4_PCA,
        "override_l5_pca": OVERRIDE_L5_PCA,
        "force_shared_pca": FORCE_SHARED_PCA,
    },
    "random_state": RANDOM_STATE,
}
with (out_dir / "run_config.json").open("w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

if SAVE_TO_SNOWFLAKE_TABLE:
    snow_df = sf_session.create_dataframe(df)
    snow_df.write.mode("overwrite").save_as_table(OUTPUT_TABLE_NAME)
    print(f"Wrote Snowflake table: {OUTPUT_TABLE_NAME}")

print(f"Saved artifacts to: {out_dir}")

Saved artifacts to: /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/analysis/lei_aws_cluster_experiment


In [22]:
# Optional: LLM-assisted naming for both L4 and L5 cluster ranges,
# including higher-level natural groupings above cluster labels.
if RUN_LLM_LABELING:
    import ast
    import re

    def _extract_json_obj(text: str) -> dict:
        """Parse JSON-like output robustly (JSON, fenced JSON, or Python dict-like)."""
        text = text.strip().replace("“", '"').replace("”", '"').replace("’", "'")

        parse_candidates = []
        parse_candidates.append(text)

        fence_match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, flags=re.DOTALL)
        if fence_match:
            parse_candidates.append(fence_match.group(1).strip())

        obj_match = re.search(r"\{[\s\S]*\}", text)
        if obj_match:
            parse_candidates.append(obj_match.group(0).strip())

        for cand in parse_candidates:
            # Strict JSON parse
            try:
                return json.loads(cand)
            except Exception:
                pass
            # Python-literal fallback (single quotes, trailing commas, etc.)
            try:
                parsed = ast.literal_eval(cand)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass

        raise ValueError("Could not parse JSON-like object from LLM response text")

    def _call_bedrock_json(prompt: str, max_tokens: int = 2400, temperature: float = 0.2) -> str:
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        }
        llm_client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)
        response = llm_client.invoke_model(
            modelId=LLM_MODEL_ID,
            body=json.dumps(body),
            contentType="application/json",
            accept="application/json",
        )
        payload = json.loads(response["body"].read().decode("utf-8"))

        content = payload.get("content", [])
        text_blocks = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text" and "text" in block:
                text_blocks.append(block["text"])
            elif isinstance(block, dict) and "text" in block:
                text_blocks.append(str(block["text"]))
            elif isinstance(block, str):
                text_blocks.append(block)

        if text_blocks:
            return "\n".join(text_blocks)

        return json.dumps(payload)

    def _normalize_hierarchy_keys(parsed: dict) -> dict:
        """Ensure required keys exist, deriving missing structures when possible."""
        parsed.setdefault("cluster_to_label", {})
        parsed.setdefault("label_descriptions", {})
        parsed.setdefault("label_to_group", {})
        parsed.setdefault("group_descriptions", {})
        parsed.setdefault("groups", {})

        # Accept groups as list and convert to object form.
        if isinstance(parsed["groups"], list):
            parsed["groups"] = {str(g): [] for g in parsed["groups"]}

        # If groups missing but label_to_group exists, derive groups.
        if (not parsed["groups"]) and parsed["label_to_group"]:
            grouped = {}
            for label, grp in parsed["label_to_group"].items():
                grouped.setdefault(str(grp), []).append(str(label))
            parsed["groups"] = grouped

        # If label_to_group missing but groups exists, derive reverse mapping.
        if (not parsed["label_to_group"]) and parsed["groups"]:
            reverse = {}
            for grp, labels in parsed["groups"].items():
                if not isinstance(labels, list):
                    continue
                for label in labels:
                    reverse[str(label)] = str(grp)
            parsed["label_to_group"] = reverse

        # Fill groups from label_to_group when groups keys exist but label lists are empty.
        if parsed["groups"] and parsed["label_to_group"]:
            for label, grp in parsed["label_to_group"].items():
                grp = str(grp)
                parsed["groups"].setdefault(grp, [])
                if str(label) not in parsed["groups"][grp]:
                    parsed["groups"][grp].append(str(label))

        return parsed

    def suggest_cluster_labels_with_bedrock(cluster_terms: dict, range_name: str):
        prompt = f"""
You are helping design product taxonomy labels for the PRODUCTS_LEI subset.
Given cluster keyword lists from the {range_name} clustering output, propose concise category labels
and natural higher-level groupings above those labels.

Rules:
- Return strict JSON only.
- Use clear business-facing labels.
- Keep labels distinct and reusable.
- Prefer <= {LLM_MAX_CATEGORIES} category labels when reasonable.
- Also create natural higher-level groups (typically ~6-10 groups, but use what fits naturally).
- Keep output concise. Do NOT include long prose.

Input:
{json.dumps(cluster_terms, indent=2)}

Return JSON object with core keys:
- cluster_to_label: object mapping cluster id (as string) -> category label
- label_to_group: object mapping category label -> high-level group name
- groups: object mapping high-level group name -> list of category labels

Optional keys:
- label_descriptions
- group_descriptions
"""

        # First attempt: direct structured JSON generation.
        raw_text = _call_bedrock_json(prompt, max_tokens=1400, temperature=LLM_TEMPERATURE)

        try:
            parsed = _extract_json_obj(raw_text)
        except Exception:
            # Save first-pass raw response for debugging.
            debug_raw_1 = out_dir / f"llm_raw_response_first_pass_{range_name.replace(' ', '_').lower()}.txt"
            debug_raw_1.write_text(raw_text, encoding="utf-8")

            # Second attempt: ask model to repair prior output into valid JSON.
            repair_prompt = f"""
Convert the following text into strictly valid JSON only.
Do not add explanations.

Required top-level keys:
- cluster_to_label
- label_descriptions
- label_to_group
- group_descriptions
- groups

Text to repair:
{raw_text}
"""
            repaired_text = _call_bedrock_json(repair_prompt, max_tokens=1800, temperature=0.0)

            debug_raw_2 = out_dir / f"llm_raw_response_repair_pass_{range_name.replace(' ', '_').lower()}.txt"
            debug_raw_2.write_text(repaired_text, encoding="utf-8")

            parsed = _extract_json_obj(repaired_text)

        parsed = _normalize_hierarchy_keys(parsed)

        required_keys = {
            "cluster_to_label",
            "label_to_group",
            "groups",
        }
        missing = sorted(required_keys - set(parsed.keys()))
        if missing:
            # Save raw output for debugging if schema still fails.
            debug_path = out_dir / f"llm_raw_response_{range_name.replace(' ', '_').lower()}.txt"
            debug_path.write_text(raw_text, encoding="utf-8")
            raise ValueError(
                f"LLM response missing required keys: {missing}. Raw response saved to {debug_path}"
            )

        return parsed

    label_suggestions_l4 = suggest_cluster_labels_with_bedrock(cluster_keywords_l4, "L4")
    label_suggestions_l5 = suggest_cluster_labels_with_bedrock(cluster_keywords_l5, "L5 exploratory")

    l4_labels_path = out_dir / f"{DATASET_SLUG}_llm_label_suggestions_l4.json"
    l5_labels_path = out_dir / f"{DATASET_SLUG}_llm_label_suggestions_l5.json"

    with l4_labels_path.open("w", encoding="utf-8") as f:
        json.dump(label_suggestions_l4, f, indent=2)
    with l5_labels_path.open("w", encoding="utf-8") as f:
        json.dump(label_suggestions_l5, f, indent=2)

    print("Saved LLM naming outputs:")
    print("-", l4_labels_path)
    print("-", l5_labels_path)

    print("\nL4 naming + groups suggestions:")
    display(label_suggestions_l4)

    print("\nL5 naming + groups suggestions:")
    display(label_suggestions_l5)
else:
    print("Set RUN_LLM_LABELING=True to generate suggested L4 and L5 category names + higher-level groups.")

ValidationException: An error occurred (ValidationException) when calling the InvokeModel operation: Invocation of model ID anthropic.claude-sonnet-4-5-20250929-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.

In [ ]:
# Publish proposed L5 labels to Snowflake from a selected LLM suggestions JSON.
# This can be re-run without re-embedding as long as df already has CLUSTER_L5_EXPLORATORY.

L5_LABELS_JSON_PATH = out_dir / f"{DATASET_SLUG}_llm_label_suggestions_l5.json"
PROPOSED_L5_TABLE_NAME = "LEI_PROPOSED_L5"  # can be fully qualified DB.SCHEMA.TABLE
PROPOSED_CLUSTER_COL = "PROPOSED_CLUSTER"
PROPOSED_LABEL_COL = "PROPOSED_LABEL"

# Optional explicit target context (recommended if table name is unqualified)
PUBLISH_DATABASE = "SNOWFLAKE_LEARNING_DB"
PUBLISH_SCHEMA = "SMCMAHON_PRODUCTS"

with open(L5_LABELS_JSON_PATH, "r", encoding="utf-8") as f:
    l5_labels_payload = json.load(f)

cluster_to_label_l5 = {
    int(k): str(v) for k, v in l5_labels_payload.get("cluster_to_label", {}).items()
}
if not cluster_to_label_l5:
    raise ValueError(f"No cluster_to_label mapping found in: {L5_LABELS_JSON_PATH}")

if "CLUSTER_L5_EXPLORATORY" not in df.columns:
    raise ValueError("CLUSTER_L5_EXPLORATORY not found in df. Re-run final assignment cell first.")

publish_df = df[["PRODUCT_ID", "CLUSTER_L5_EXPLORATORY"]].copy()
publish_df[PROPOSED_CLUSTER_COL] = publish_df["CLUSTER_L5_EXPLORATORY"].astype("int32")
publish_df[PROPOSED_LABEL_COL] = publish_df[PROPOSED_CLUSTER_COL].map(cluster_to_label_l5)

unmapped = sorted(publish_df.loc[publish_df[PROPOSED_LABEL_COL].isna(), PROPOSED_CLUSTER_COL].unique())
if unmapped:
    raise ValueError(f"Missing labels for clusters: {unmapped}")

publish_df = publish_df[["PRODUCT_ID", PROPOSED_CLUSTER_COL, PROPOSED_LABEL_COL]]

def _parse_qualified_name(name: str):
    parts = [p.strip() for p in str(name).split(".") if p.strip()]
    if len(parts) == 3:
        return parts[0], parts[1], parts[2]
    if len(parts) == 1:
        return None, None, parts[0]
    raise ValueError(f"Expected table name as TABLE or DB.SCHEMA.TABLE, got: {name}")


target_db, target_schema, target_table = _parse_qualified_name(PROPOSED_L5_TABLE_NAME)

# Resolve DB/SCHEMA context for write_pandas temp stage creation.
if PUBLISH_DATABASE:
    target_db = PUBLISH_DATABASE
if PUBLISH_SCHEMA:
    target_schema = PUBLISH_SCHEMA

if not target_db or not target_schema:
    # Fall back to source table context when publish target is unqualified.
    src_db, src_schema, _ = _parse_qualified_name(TABLE)
    target_db = target_db or src_db
    target_schema = target_schema or src_schema

if not target_db or not target_schema:
    raise ValueError(
        "Could not resolve target database/schema. Set PUBLISH_DATABASE and PUBLISH_SCHEMA explicitly."
    )

sf_session.sql(f"USE DATABASE {target_db}").collect()
sf_session.sql(f"USE SCHEMA {target_schema}").collect()

full_target_name = f"{target_db}.{target_schema}.{target_table}"

snow_publish_df = sf_session.create_dataframe(publish_df)
snow_publish_df.write.mode("overwrite").save_as_table(full_target_name)

print(f"Wrote Snowflake table: {full_target_name}")
print(publish_df.head())

Wrote Snowflake table: SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PROPOSED_L5
           PRODUCT_ID  PROPOSED_CLUSTER         PROPOSED_LABEL
0  01tPP00000D4F2uYAF                37            Microplates
1  01tPP00000DKnYoYAL                 2     Laboratory Bottles
2  01tPK00000GcaFeYAJ                45  Cables and Connectors
3  01tPK00000GcaPeYAJ                33  Lab Storage Solutions
4  01tPP00000D4SUXYA3                26            Amino Acids


In [ ]:
print("df in memory:", "df" in globals())
if "df" in globals():
    print("df shape:", df.shape)

print("CLUSTER_L5_EXPLORATORY column exists:",
      "df" in globals() and "CLUSTER_L5_EXPLORATORY" in df.columns)

df in memory: True
df shape: (450405, 18)
CLUSTER_L5_EXPLORATORY column exists: True
